In [120]:
import snowflake.connector
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv(dotenv_path="/Users/pararthivellanki/airflow/.env")

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA")
)

cur = conn.cursor()
cur.execute("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA()")
print(cur.fetchall())

[('TASTY_BYTES', 'PUBLIC')]


In [43]:
# !pip3 install snowflake-connector-python
#!pip3 install python-dotenv
#!pip3 install "snowflake-connector-python[pandas]"

In [121]:
cur.execute("USE DATABASE TEST_DB")
cur.execute("USE SCHEMA test_schema")

In [89]:
print(cur.fetchall())

[('Statement executed successfully.',)]


In [122]:
cur.execute("SHOW TABLES")
tables = cur.fetchall()
for table in tables:
    print(table[1])  # index 1 is the table name

DAILY_PLAN
DATA
PLAYER_LOG
TEST_TABLE
TEST_TABLE2


In [123]:
cur.execute("SELECT * FROM DAILY_PLAN")

In [124]:
def results_to_dataframe(cursor):
    rows = cursor.fetchall()
    cols = [desc[0] for desc in cursor.description]
    return pd.DataFrame(rows, columns=cols)

In [125]:
results_df = results_to_dataframe(cur)
print(results_df.head())

  DAY_ID   DAY_NAME                         WORKOUT  \
0     D1     Monday  Strength Training - Upper Body   
1     D2    Tuesday      Cardio - 5km Run + Cycling   
2     D3  Wednesday  Strength Training - Lower Body   
3     D4   Thursday          Agility & Speed Drills   
4     D5     Friday                  Full Body HIIT   

                                                DIET  
0  High Protein: Chicken, Eggs, Whey Shake, Brown...  
1      High Carbs: Oats, Banana, Sweet Potato, Pasta  
2  High Protein: Fish, Lentils, Greek Yogurt, Quinoa  
3  Balanced: Lean Meat, Vegetables, Nuts, Whole G...  
4     High Carbs + Protein: Rice, Eggs, Fruits, Milk  


In [110]:
view = " create or replace view analytics.active_accounts as select * from test_schema.data where insurance_status = 'Active' "

In [111]:
cur.execute("USE SCHEMA analytics")
cur.execute(view)

In [116]:
cur.execute("show views")
cur.fetchall()



[(datetime.datetime(2026, 6, 6, 20, 0, 5, 397000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'ACTIVE_ACCOUNTS',
  '',
  'TEST_DB',
  'ANALYTICS',
  'ACCOUNTADMIN',
  '',
  "create or replace view analytics.active_accounts as select * from test_schema.data where insurance_status = 'Active' ",
  'false',
  'false',
  'ROLE',
  'OFF')]

In [118]:
cur.execute("select * from analytics.active_accounts")
data = results_to_dataframe(cur)    
print(data.head())

   CLAIM_DATE CUSTOMER_ID  CUSTOMER_ZIP_CODE DB_UPDATE_DATE  \
0  19/09/2007   J12676787             2614.0     30/10/2011   
1  19/06/2009   J13023604             2196.0      2/04/2011   
2  26/07/2009   J13028720             4125.0     31/08/2011   
3         NaN   J12725766             5072.0      5/12/2011   
4         NaN   J12668156             2557.0     10/10/2011   

  INSURANCE_EXPIRY_DATE INSURANCE_STATUS MODEL_LINE ORIGINAL_SOURCE  \
0            26/10/2012           Active     SL-300           State   
1            20/03/2012           Active       Yard           State   
2            22/06/2012           Active       Yard    State-Public   
3             5/05/2012           Active     SL-500           State   
4            12/05/2012           Active     SL-300           State   

  PRODUCT_PRODUCTION_DATE      STATUS SUB_MODEL_LINE  SUB_STATUS  VENDOR_ID  \
0               5/03/1997    COMPLETE     Dual Phase    Complete   200284.0   
1              26/06/1998    COMPLET